# Mandi360: Data Mining and Predictive Analysis Notebook

This notebook documents and reruns the project code used for the Data Mining and Predictive Analysis parts of Mandi360.

It uses the existing project modules instead of duplicate notebook-only logic, so the notebook and Streamlit dashboard stay consistent.

## What This Notebook Covers

- **Data Mining:** review grouping with TF-IDF + KMeans clustering.
- **Data Mining:** association-rule mining to find issues that appear together.
- **Predictive Analysis:** monthly branch early-warning detection.
- **Predictive Analysis:** branch rating forecast and management attention level.

Raw data files are not modified. Important outputs are written under `results/`.

In [ ]:
from __future__ import annotations

import csv
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analytics.anomaly import build_monthly_branch_features, detect_anomalies, write_anomaly_artifacts
from src.analytics.association import mine_association_rules, write_association_artifacts
from src.analytics.clustering import fit_review_clusters, write_cluster_artifacts
from src.analytics.forecasting import forecast_branch_ratings, write_forecast_artifacts
from src.config.settings import ANOMALY_RESULTS_DIR, ASSOCIATION_RESULTS_DIR, CLUSTERING_RESULTS_DIR, FORECAST_RESULTS_DIR, INTERIM_DIR, NLP_RESULTS_DIR

pd.set_option("display.max_colwidth", 120)
ROOT

## Load Prepared Review Data

The notebook reads prepared project outputs from `data/interim/` and `results/tables/nlp/`. If these files are missing, run the main pipeline first with `python -m src.pipeline`.

In [ ]:
def read_csv_records(path: Path) -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    with path.open("r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def to_float(value):
    if value in (None, "", "nan"):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


processed_records = read_csv_records(INTERIM_DIR / "reviews_preprocessed.csv")
nlp_records = read_csv_records(NLP_RESULTS_DIR / "reviews_nlp_baseline.csv")
aspect_rows = read_csv_records(NLP_RESULTS_DIR / "review_aspects_baseline.csv")

for row in nlp_records:
    row["rating"] = to_float(row.get("rating"))
    row["sentiment_score"] = to_float(row.get("sentiment_score"))

print(f"Prepared reviews: {len(processed_records):,}")
print(f"Reviews with customer mood fields: {len(nlp_records):,}")
print(f"Experience-area mentions: {len(aspect_rows):,}")

## Data Mining 1: Review Groups

This step groups similar review text using TF-IDF features and KMeans clustering. The result is used as a discovery aid, not as final business labels.

In [ ]:
cluster_model = fit_review_clusters(processed_records)
write_cluster_artifacts(cluster_model, CLUSTERING_RESULTS_DIR)

cluster_summary = pd.DataFrame(cluster_model["clusters"])
cluster_summary["customer group"] = [f"Group {index}" for index in range(1, len(cluster_summary) + 1)]
cluster_summary["common customer words"] = cluster_summary["top_terms"].apply(lambda terms: ", ".join(terms[:8]))
cluster_summary = cluster_summary[["customer group", "size", "common customer words"]].rename(columns={"size": "reviews in group"})

print(f"Review groups found: {cluster_model['params']['selected_k']}")
print(f"Reviews used: {cluster_model['params']['documents_used']:,}")
display(cluster_summary)

## Data Mining 2: Issues That Appear Together

Association rules identify combinations of rating bucket, customer mood, and experience areas that often appear in the same review.

In [ ]:
def format_item(item: str) -> str:
    if item.startswith("aspect:"):
        return "Experience: " + item.split(":", 1)[1].replace("_", " ").title()
    if item.startswith("sentiment:"):
        return "Customer mood: " + item.split(":", 1)[1]
    return {
        "rating_low": "Low rating (1-2)",
        "rating_mid": "Neutral rating (3)",
        "rating_high": "High rating (4-5)",
    }.get(item, item.replace("_", " ").title())


association_result = mine_association_rules(nlp_records, aspect_rows)
write_association_artifacts(association_result, ASSOCIATION_RESULTS_DIR)

rules_df = pd.DataFrame(association_result["rules"])
if rules_df.empty:
    print("No association rules met the current support and reliability thresholds.")
else:
    rules_df["when customers mention"] = rules_df["antecedent"].apply(lambda items: ", ".join(format_item(item) for item in items))
    rules_df["they also mention"] = rules_df["consequent"].apply(lambda items: ", ".join(format_item(item) for item in items))
    display(rules_df[["when customers mention", "they also mention", "support", "confidence", "lift"]].head(15))

print(f"Rules found: {association_result['rule_count']:,}")

## Predictive Analysis 1: Monthly Branch Early-Warning View

This step aggregates each branch by month and flags unusual combinations of review volume, average rating, customer mood, and negative feedback share.

In [ ]:
monthly_branch_features = build_monthly_branch_features(nlp_records)
anomaly_result = detect_anomalies(monthly_branch_features)
write_anomaly_artifacts(anomaly_result, ANOMALY_RESULTS_DIR)

attention_rows = [row for row in anomaly_result["rows"] if row["is_anomaly"]]
attention_df = pd.DataFrame(attention_rows)
if attention_df.empty:
    print("No unusual branch-months were flagged.")
else:
    attention_df = attention_df[["branch_name", "month", "alert_severity", "alert_reasons", "review_count", "average_rating", "negative_ratio"]]
    attention_df = attention_df.rename(columns={
        "branch_name": "branch",
        "alert_severity": "urgency",
        "alert_reasons": "why it needs attention",
        "review_count": "reviews",
        "average_rating": "average rating",
        "negative_ratio": "negative feedback share",
    })
    display(attention_df.head(20))

print(f"Months analyzed: {len(anomaly_result['rows']):,}")
print(f"Months needing attention: {anomaly_result['anomaly_count']:,}")

## Predictive Analysis 2: Branch Rating Forecast

This step fits a transparent linear trend per branch and forecasts the next few months. Forecasts are directional planning signals, not guarantees.

In [ ]:
forecast_result = forecast_branch_ratings(monthly_branch_features)
write_forecast_artifacts(forecast_result, FORECAST_RESULTS_DIR)

forecast_rows = []
for branch in forecast_result["branches_forecasted"]:
    branch_name = branch["branch_id"].replace("_", " ").title()
    for row in branch["forecast"]:
        forecast_rows.append({
            "branch": branch_name,
            "management attention": branch["risk_level"].title(),
            "expected monthly direction": branch["trend_slope_per_month"],
            "month": row["month"],
            "expected rating": row["predicted_average_rating"],
            "lower expected range": row["lower_bound"],
            "upper expected range": row["upper_bound"],
        })

forecast_df = pd.DataFrame(forecast_rows)
display(forecast_df)

if forecast_result["branches_skipped_insufficient_history"]:
    skipped = pd.DataFrame(forecast_result["branches_skipped_insufficient_history"])
    skipped["branch"] = skipped["branch_id"].str.replace("_", " ").str.title()
    display(skipped[["branch", "months_available"]])

## Assumptions and Limitations

- Review text and ratings come from collected Google Maps exports already prepared by the project pipeline.
- Clustering and association rules are discovery tools; they do not prove cause and effect.
- Early-warning flags are analytical signals requiring manager review.
- Forecasts use short monthly history, so they should be treated as directional planning support, not certainty.
- Raw data files are not changed by this notebook.